<a href="https://colab.research.google.com/github/saim9211/DeepLearning/blob/main/07_pytorch_experiment_tracking_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. PyTorch Experiment Tracking Exercise Template

Welcome to the 07. PyTorch Experiment Tracking exercise template notebook.

> **Note:** There may be more than one solution to each of the exercises. This notebook only shows one possible example.

## Resources

1. These exercises/solutions are based on [section 07. PyTorch Transfer Learning](https://www.learnpytorch.io/07_pytorch_experiment_tracking/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.
2. See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/cO_r2FYcAjU).
3. See [other solutions on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions).

> **Note:** The first section of this notebook is dedicated to getting various helper functions and datasets used for the exercises. The exercises start at the heading "Exercise 1: ...".

### Get various imports and helper functions

We'll need to make sure we have `torch` v.1.12+ and `torchvision` v0.13+.

In [1]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U --pre torch torchvision --extra-index-url https://download.pytorch.org/whl/nightly/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")

[INFO] torch/torchvision versions not as required, installing nightly versions.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/nightly/cu113
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 61.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 70.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 99.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 50.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 76.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 117.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 175.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 33.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 130.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 208.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━

In [ ]:
 # Make sure we have a GPU
 import torch
 device = "cuda" if torch.cuda.is_available() else "cpu"
 device

In [ ]:
# Get regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

In [ ]:
# Set seeds
def set_seeds(seed: int=42):
    """Sets random sets for torch operations.

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    # Set the seed for general torch operations
    torch.manual_seed(seed)
    # Set the seed for CUDA torch operations (ones that happen on the GPU)
    torch.cuda.manual_seed(seed)

In [ ]:
import os
import zipfile

from pathlib import Path

import requests

def download_data(source: str,
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.

    Returns:
        pathlib.Path to downloaded data.

    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                      destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it...
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)

        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...")
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)

    return image_path

image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
from torch.utils.tensorboard import SummaryWriter
def create_writer(experiment_name: str,
                  model_name: str,
                  extra: str=None):
    """Creates a torch.utils.tensorboard.writer.SummaryWriter() instance saving to a specific log_dir.

    log_dir is a combination of runs/timestamp/experiment_name/model_name/extra.

    Where timestamp is the current date in YYYY-MM-DD format.

    Args:
        experiment_name (str): Name of experiment.
        model_name (str): Name of model.
        extra (str, optional): Anything extra to add to the directory. Defaults to None.

    Returns:
        torch.utils.tensorboard.writer.SummaryWriter(): Instance of a writer saving to log_dir.

    Example usage:
        # Create a writer saving to "runs/2022-06-04/data_10_percent/effnetb2/5_epochs/"
        writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb2",
                               extra="5_epochs")
        # The above is the same as:
        writer = SummaryWriter(log_dir="runs/2022-06-04/data_10_percent/effnetb2/5_epochs/")
    """
    from datetime import datetime
    import os

    # Get timestamp of current date (all experiments on certain day live in same folder)
    timestamp = datetime.now().strftime("%Y-%m-%d") # returns current date in YYYY-MM-DD format

    if extra:
        # Create log directory path
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)

    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

In [ ]:
# Create a test writer
writer = create_writer(experiment_name="test_experiment_name",
                       model_name="this_is_the_model_name",
                       extra="add_a_little_extra_if_you_want")

In [ ]:
from typing import Dict, List
from tqdm.auto import tqdm

from going_modular.going_modular.engine import train_step, test_step

# Add writer parameter to train()
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          writer: torch.utils.tensorboard.writer.SummaryWriter # new parameter to take in a writer
          ) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Stores metrics to specified writer log_dir if present.

    Args:
      model: A PyTorch model to be trained and tested.
      train_dataloader: A DataLoader instance for the model to be trained on.
      test_dataloader: A DataLoader instance for the model to be tested on.
      optimizer: A PyTorch optimizer to help minimize the loss function.
      loss_fn: A PyTorch loss function to calculate loss on both datasets.
      epochs: An integer indicating how many epochs to train for.
      device: A target device to compute on (e.g. "cuda" or "cpu").
      writer: A SummaryWriter() instance to log model results to.

    Returns:
      A dictionary of training and testing loss as well as training and
      testing accuracy metrics. Each metric has a value in a list for
      each epoch.
      In the form: {train_loss: [...],
                train_acc: [...],
                test_loss: [...],
                test_acc: [...]}
      For example if training for epochs=2:
              {train_loss: [2.0616, 1.0537],
                train_acc: [0.3945, 0.3945],
                test_loss: [1.2641, 1.5706],
                test_acc: [0.3400, 0.2973]}
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        # Print out what's happening
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)


        ### New: Use the writer parameter to track experiments ###
        # See if there's a writer, if so, log to it
        if writer:
            # Add results to SummaryWriter
            writer.add_scalars(main_tag="Loss",
                               tag_scalar_dict={"train_loss": train_loss,
                                                "test_loss": test_loss},
                               global_step=epoch)
            writer.add_scalars(main_tag="Accuracy",
                               tag_scalar_dict={"train_acc": train_acc,
                                                "test_acc": test_acc},
                               global_step=epoch)

            # Close the writer
            writer.close()
        else:
            pass
    ### End new ###

    # Return the filled results at the end of the epochs
    return results

### Download data

Using the same data from https://www.learnpytorch.io/07_pytorch_experiment_tracking/

In [ ]:
# Download 10 percent and 20 percent training data (if necessary)
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

In [ ]:
# Setup training directory paths
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

# Setup testing directory paths (note: use the same test dataset for both to compare the results)
test_dir = data_10_percent_path / "test"

# Check the directories
print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

In [ ]:
from torchvision import transforms

# Create a transform to normalize data distribution to be inline with ImageNet
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], # values per colour channel [red, green, blue]
                                 std=[0.229, 0.224, 0.225])

# Create a transform pipeline
simple_transform = transforms.Compose([
                                       transforms.Resize((224, 224)),
                                       transforms.ToTensor(), # get image values between 0 & 1
                                       normalize
])

### Turn data into DataLoaders

In [ ]:
# Note: Data augmentation transform like this should only be performed on training data
train_transform_data_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    normalize
])

# Create a helper function to visualize different augmented (and not augmented) images
def view_dataloader_images(dataloader, n=10):
    if n > 10:
        print(f"Having n higher than 10 will create messy plots, lowering to 10.")
        n = 10
    imgs, labels = next(iter(dataloader))
    plt.figure(figsize=(16, 8))
    for i in range(n):
        # Min max scale the image for display purposes
        targ_image = imgs[i]
        sample_min, sample_max = targ_image.min(), targ_image.max()
        sample_scaled = (targ_image - sample_min)/(sample_max - sample_min)

        # Plot images with appropriate axes information
        plt.subplot(1, 10, i+1)
        plt.imshow(sample_scaled.permute(1, 2, 0)) # resize for Matplotlib requirements
        plt.title(class_names[labels[i]])
        plt.axis(False)

print(f"Data augmentation transform created: {train_transform_data_aug}")


In [ ]:
import os
from torch.utils.data import DataLoader
from torchvision import datasets

NUM_WORKERS = os.cpu_count() # use maximum number of CPUs for workers to load data

# Note: this is an update version of data_setup.create_dataloaders to handle
# differnt train and test transforms.
def create_dataloaders(
    train_dir,
    test_dir,
    train_transform, # add parameter for train transform (transforms on train dataset)
    test_transform,  # add parameter for test transform (transforms on test dataset)
    batch_size=32, num_workers=NUM_WORKERS
):
    # Use ImageFolder to create dataset(s)
    train_data = datasets.ImageFolder(train_dir, transform=train_transform)
    test_data = datasets.ImageFolder(test_dir, transform=test_transform)

    # Get class names
    class_names = train_data.classes

    # Turn images into data loaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False, # No need to shuffle test data
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names

print("Updated `create_dataloaders` function defined.")


In [ ]:
BATCH_SIZE = 32

# Create 10% training and test DataLoaders
train_dataloader_10_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_10_percent,
                                                                                          test_dir=test_dir,
                                                                                          transform=simple_transform,
                                                                                          batch_size=BATCH_SIZE)

# Create 20% training and test DataLoaders
train_dataloader_20_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_20_percent,
                                                                                          test_dir=test_dir,
                                                                                          transform=simple_transform,
                                                                                          batch_size=BATCH_SIZE)

# Find the number of samples/batches per dataloader (using the same test_dataloader for both experiments)
print(f"Number of batches of size {BATCH_SIZE} in 10 percent training data: {len(train_dataloader_10_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in 20 percent training data: {len(train_dataloader_20_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in testing data: {len(train_dataloader_10_percent)} (all experiments will use the same test set)")
print(f"Number of classes: {len(class_names)}, class names: {class_names}")

In [ ]:
# Create 20% training and test DataLoaders with data augmentation
train_dataloader_20_percent_aug, test_dataloader_20_percent_aug, class_names = create_dataloaders(train_dir=train_dir_20_percent,
                                                                                                   test_dir=test_dir,
                                                                                                   train_transform=train_transform_data_aug,
                                                                                                   test_transform=simple_transform,
                                                                                                   batch_size=BATCH_SIZE)

print(f"Number of batches of size {BATCH_SIZE} in 20 percent augmented training data: {len(train_dataloader_20_percent_aug)}")


Let's visualize some augmented images.

In [ ]:
view_dataloader_images(train_dataloader_20_percent_aug, n=10)


## Exercise 1: Pick a larger model from [`torchvision.models`](https://pytorch.org/vision/main/models.html) to add to the list of experiments (for example, EffNetB3 or higher)

* How does it perform compared to our existing models?
* **Hint:** You'll need to set up an exerpiment similar to [07. PyTorch Experiment Tracking section 7.6](https://www.learnpytorch.io/07_pytorch_experiment_tracking/#76-create-experiments-and-set-up-training-code).

In [ ]:
import torch
from torch import nn # Needed for the model definitions
import torchvision # Needed for the model definitions

# Define create_effnetb2 and create_effnetb3 functions here
# This is done to explicitly define them in case they are missing from model_builder.py
# from the downloaded going_modular package, which which seems to be causing an ImportError.

def create_effnetb2(num_classes: int=3,
                    seed: int=42):
    """
    Creates an EfficientNetB2 feature extractor model and sets its
    top layer to `num_classes`.

    Args:
        num_classes (int): Number of classes in the classifier.
        seed (int, optional): Random seed for reproducible creation. Defaults to 42.
    """
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    model = torchvision.models.efficientnet_b2(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )
    return model

def create_effnetb3(num_classes: int=3,
                    seed: int=42):
    """
    Creates an EfficientNetB3 feature extractor model and sets its
    top layer to `num_classes`.

    Args:
        num_classes (int): Number of classes in the classifier.
        seed (int, optional): Random seed for reproducible creation. Defaults to 42.
    """
    weights = torchvision.models.EfficientNet_B3_Weights.DEFAULT
    model = torchvision.models.efficientnet_b3(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1536, out_features=num_classes)
    )
    return model


%time
from going_modular.going_modular.utils import save_model

set_seeds(42)
experiment_number=0

# Define experiment configurations
experiment_configs = [
    # Exercise 1: Baseline experiments with 10% and 20% data, 5 and 10 epochs
    {'model_type': 'effnetb2', 'dataloader': train_dataloader_10_percent, 'dataloader_desc': '10_percent_data', 'epochs': 5},
    {'model_type': 'effnetb2', 'dataloader': train_dataloader_10_percent, 'dataloader_desc': '10_percent_data', 'epochs': 10},
    {'model_type': 'effnetb3', 'dataloader': train_dataloader_20_percent, 'dataloader_desc': '20_percent_data', 'epochs': 5},
    {'model_type': 'effnetb3', 'dataloader': train_dataloader_20_percent, 'dataloader_desc': '20_percent_data', 'epochs': 10},
    # Exercise 2: New experiments with data augmentation on 20% data, 5 and 10 epochs
    {'model_type': 'effnetb2', 'dataloader': train_dataloader_20_percent_aug, 'dataloader_desc': '20_percent_data_aug', 'epochs': 5},
    {'model_type': 'effnetb2', 'dataloader': train_dataloader_20_percent_aug, 'dataloader_desc': '20_percent_data_aug', 'epochs': 10},
    {'model_type': 'effnetb3', 'dataloader': train_dataloader_20_percent_aug, 'dataloader_desc': '20_percent_data_aug', 'epochs': 5},
    {'model_type': 'effnetb3', 'dataloader': train_dataloader_20_percent_aug, 'dataloader_desc': '20_percent_data_aug', 'epochs': 10},
]

print("\n--- Starting experiments ---\n")

# Iterate through experiment configurations
for experiment_config in experiment_configs:
    experiment_number += 1
    current_model_type = experiment_config['model_type']
    current_dataloader = experiment_config['dataloader']
    current_dataloader_desc = experiment_config['dataloader_desc']
    epoch = experiment_config['epochs']

    print(f"[INFO] Experiment number: {experiment_number}")
    print(f"[INFO] Model: {current_model_type}")
    print(f"[INFO] DataLoader: {current_dataloader_desc}")
    print(f"[INFO] Number of epochs: {epoch}")

    # Create a writer for each experiment run (before training)
    writer = create_writer(
        experiment_name=current_dataloader_desc, # Use dataloader description for experiment name
        model_name=current_model_type, # Use current model type for model name
        extra=f"epochs_{epoch}"
    )

    if current_model_type=='effnetb2':
        model=create_effnetb2(num_classes=len(class_names))
    elif current_model_type=='effnetb3':
        model=create_effnetb3(num_classes=len(class_names))
    else:
        print(f"[ERROR] Unknown model type: {current_model_type}")
        continue # Skip this experiment if model type is unknown

    model.to(device)
    loss_fn=nn.CrossEntropyLoss()
    optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
    res=train(model=model,
              train_dataloader=current_dataloader,
              test_dataloader=test_dataloader,
              optimizer=optimizer,
              loss_fn=loss_fn,
              epochs=epoch,
              device=device,
              writer=writer) # Pass the newly created writer

    # Note: The 'train' function (defined in VwO0Q1eFsusV) calls writer.close()
    # after each epoch, which means only the first epoch's data will be logged
    # if you intend to log data for all epochs to the same writer.
    # If you want to log all epochs, the writer.close() call might need to be
    # moved out of the train function (e.g., after the train() call in this loop).


In [ ]:

#!pip install --upgrade setuptools tensorboard
%load_ext tensorboard
%tensorboard --logdir runs


## Exercise 2. Introduce data augmentation to the list of experiments using the 20% pizza, steak, sushi training and test datasets, does this change anything?
    
* For example, you could have one training DataLoader that uses data augmentation (e.g. `train_dataloader_20_percent_aug` and `train_dataloader_20_percent_no_aug`) and then compare the results of two of the same model types training on these two DataLoaders.
* **Note:** You may need to alter the `create_dataloaders()` function to be able to take a transform for the training data and the testing data (because you don't need to perform data augmentation on the test data). See [04. PyTorch Custom Datasets section 6](https://www.learnpytorch.io/04_pytorch_custom_datasets/#6-other-forms-of-transforms-data-augmentation) for examples of using data augmentation or the script below for an example:

```python
# Note: Data augmentation transform like this should only be performed on training data
train_transform_data_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    normalize
])

# Create a helper function to visualize different augmented (and not augmented) images
def view_dataloader_images(dataloader, n=10):
    if n > 10:
        print(f"Having n higher than 10 will create messy plots, lowering to 10.")
        n = 10
    imgs, labels = next(iter(dataloader))
    plt.figure(figsize=(16, 8))
    for i in range(n):
        # Min max scale the image for display purposes
        targ_image = imgs[i]
        sample_min, sample_max = targ_image.min(), targ_image.max()
        sample_scaled = (targ_image - sample_min)/(sample_max - sample_min)

        # Plot images with appropriate axes information
        plt.subplot(1, 10, i+1)
        plt.imshow(sample_scaled.permute(1, 2, 0)) # resize for Matplotlib requirements
        plt.title(class_names[labels[i]])
        plt.axis(False)

# Have to update `create_dataloaders()` to handle different augmentations
import os
from torch.utils.data import DataLoader
from torchvision import datasets

NUM_WORKERS = os.cpu_count() # use maximum number of CPUs for workers to load data

# Note: this is an update version of data_setup.create_dataloaders to handle
# differnt train and test transforms.
def create_dataloaders(
    train_dir,
    test_dir,
    train_transform, # add parameter for train transform (transforms on train dataset)
    test_transform,  # add parameter for test transform (transforms on test dataset)
    batch_size=32, num_workers=NUM_WORKERS
):
    # Use ImageFolder to create dataset(s)
    train_data = datasets.ImageFolder(train_dir, transform=train_transform)
    test_data = datasets.ImageFolder(test_dir, transform=test_transform)

    # Get class names
    class_names = train_data.classes

    # Turn images into data loaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names
```

In [ ]:
# TODO: your code


## Exercise 3. Scale up the dataset to turn FoodVision Mini into FoodVision Big using the entire [Food101 dataset from `torchvision.models`](https://pytorch.org/vision/stable/generated/torchvision.datasets.Food101.html#torchvision.datasets.Food101)
    
* You could take the best performing model from your various experiments or even the EffNetB2 feature extractor we created in this notebook and see how it goes fitting for 5 epochs on all of Food101.
* If you try more than one model, it would be good to have the model's results tracked.
* If you load the Food101 dataset from `torchvision.models`, you'll have to create PyTorch DataLoaders to use it in training.
* **Note:** Due to the larger amount of data in Food101 compared to our pizza, steak, sushi dataset, this model will take longer to train.

In [1]:
import torchvision
from torchvision.datasets import Food101
from torch.utils.data import DataLoader
from torchvision import transforms

normalize=transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])
train_dataset=Food101(
    root="./data",
    split="train",
    transform=transform,
    download=True
)
test_dataset=Food101(
    root="./data",
    split="test",
    transform=transform,
    download=True
)
# now dataloader

train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=True)

#
img,label=next(iter(train_loader))
print(img.shape, label.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32])


In [2]:
print(train_dataset.classes)

['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheese_plate', 'cheesecake', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras', 'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice', 'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich', 'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup', 'hot_dog', 'huevos_rancheros', 'hummus', 'ice_cream', 'lasagna', 'lobster_bisque', 'lobster_roll_sandwich', 'macaroni_and_cheese', 'macarons', 'miso_sou

In [3]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision.datasets import Food101
from torchvision import transforms


set the configration

In [4]:
device="cuda"if torch.cuda.is_available else "cpu"
device

'cuda'

In [5]:
BATCH_SIZE=32
Epoch=5


In [6]:
#automatic transform
weight=torchvision.models.EfficientNet_B0_Weights.DEFAULT
transform_w=weight.transforms()
transform_w

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [7]:
train_data=Food101(
    root="./data",
    split="train",
    transform=transform_w,
    download=False
)
test_data=Food101(
    root="./data",
    split="test",
    transform=transform_w,
    download=False
)
print(len(train_data))
print(len(test_data))

75750
25250


In [8]:
#dataloader
train_data_loader=DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True)
test_data_loader=DataLoader(test_data,batch_size=BATCH_SIZE,shuffle=True)
train_data_loader,test_data_loader

(<torch.utils.data.dataloader.DataLoader at 0x7e6eb6de9a90>,
 <torch.utils.data.dataloader.DataLoader at 0x7e6fc10ed5b0>)

In [9]:
model=torchvision.models.efficientnet_b0(weights=weight).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [10]:
for param in model.parameters():
    param.requires_grad=False
model.classifier=nn.Sequential(
    nn.Dropout(p=0.3,inplace=True),
    nn.Linear(in_features=1280,out_features=101)
).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [11]:
!pip install torchinfo

In [12]:
from torchinfo import summary
summary(model,input_size=(1,3,224,224),
        col_names=[
            "input_size",
            "output_size",
            "num_params",
            "trainable",
        ])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [1, 3, 224, 224]          [1, 101]                  --                        Partial
├─Sequential: 1-1                                       [1, 3, 224, 224]          [1, 1280, 7, 7]           --                        False
│    └─Conv2dNormActivation: 2-1                        [1, 3, 224, 224]          [1, 32, 112, 112]         --                        False
│    │    └─Conv2d: 3-1                                 [1, 3, 224, 224]          [1, 32, 112, 112]         (864)                     False
│    │    └─BatchNorm2d: 3-2                            [1, 32, 112, 112]         [1, 32, 112, 112]         (64)                      False
│    │    └─SiLU: 3-3                                   [1, 32, 112, 112]         [1, 32, 112, 112]         --                        --
│    └─Sequential

complete parameters report

In [13]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Frozen Parameters    : {frozen_params:,}")

Total Parameters     : 4,136,929
Trainable Parameters : 129,381
Frozen Parameters    : 4,007,548


In [14]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)


In [16]:
val_losses = []
val_accuracies = []
train_losses = []
train_accuracies = []

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
EPOCHS=2

for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)

        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total


    # =========================
    # EVALUATION
    # =========================

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = loss_fn(outputs, labels)

            running_val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = running_val_loss / len(test_loader)
    val_acc = 100 * val_correct / val_total

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.2f}% "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.2f}%"
    )

Epoch [1/2] | Train Loss: 2.1695 | Train Acc: 46.91% | Val Loss: 1.7298 | Val Acc: 56.21%
Epoch [2/2] | Train Loss: 2.0869 | Train Acc: 48.68% | Val Loss: 1.7176 | Val Acc: 56.59%


In [1]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [4]:
import mlflow
import mlflow.pytorch

In [7]:
import os

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_experiment("Food101_EfficientNetB0")

2026/08/31 18:04:33 INFO mlflow.tracking.fluent: Experiment with name 'Food101_EfficientNetB0' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///content/mlruns/672852835390998665', creation_time=1788199473782, effective_trace_archival_retention=None, experiment_id='672852835390998665', last_update_time=1788199473782, lifecycle_stage='active', name='Food101_EfficientNetB0', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run(run_name="EfficientNetB0_5_Epochs"):

    # Parameters
    mlflow.log_params({
        "model": "EfficientNet-B0",
        "dataset": "Food-101",
        "epochs": 5,
        "batch_size": BATCH_SIZE,

    })

    # Log metrics for every epoch
    for epoch in range(5):

        mlflow.log_metrics({
            "train_loss": train_losses[epoch],
            "train_accuracy": train_accuracies[epoch],
            "val_loss": val_losses[epoch],
            "val_accuracy": val_accuracies[epoch]
        }, step=epoch + 1)

    # Final metrics
    mlflow.log_metrics({
        "final_train_accuracy": train_accuracies[-1],
        "final_val_accuracy": val_accuracies[-1],
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1]
    })

    print("MLflow experiment logged successfully.")


In [ ]:
    mlflow.pytorch.log_model(
        model,
        name="efficientnet_b0_food101",
        input_example=torch.randn(1, 3, 224, 224).to(device),
        serialization_format='pickle' # Set serialization format to 'pickle'
    )

In [ ]:
!pip install -q mlflow pyngrok

In [ ]:
import os

os.environ["NGROK_AUTHTOKEN"] = "3IeeCelPs5fr6OjKeSQ0ZeenuHu_6bSEdrmZbeBaxPp6x1TNB"

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

print("ngrok authentication successful")

In [ ]:
import mlflow
import os
import shutil # Import shutil for rmtree

MLFLOW_DIR = "/content/mlruns"

# Remove the MLflow directory to ensure a clean state, if it exists
if os.path.exists(MLFLOW_DIR):
    shutil.rmtree(MLFLOW_DIR)
    print(f"[INFO] Removed existing MLflow tracking directory: {MLFLOW_DIR}")

os.makedirs(MLFLOW_DIR, exist_ok=True)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true" # Allow file store

mlflow.set_tracking_uri(
    "file:///content/mlruns"
)

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

In [ ]:
import os
from mlflow.tracking import MlflowClient

# os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true" # Removed from here

client = MlflowClient(
    tracking_uri="file:///content/mlruns"
)

experiments = client.search_experiments()

for exp in experiments:
    print(
        "ID:", exp.experiment_id,
        "| Name:", exp.name
    )

In [ ]:
import os

# Ensure the .trash directory exists to prevent MLflow file store errors
try:
    trash_path = os.path.join("/content/mlruns", ".trash")
    if not os.path.isdir(trash_path):
        os.makedirs(trash_path, exist_ok=True)
        print(f"[INFO] Created missing MLflow .trash directory: {trash_path}")
except Exception as e:
    print(f"[WARNING] Could not ensure MLflow .trash directory exists: {e}")

runs = client.search_runs(
    experiment_ids=["1"]
)

for run in runs:

    print("Run ID:", run.info.run_id)

    print(
        "Validation Accuracy:",
        run.data.metrics.get("final_val_accuracy")
    )

In [ ]:
!pkill -f "mlflow ui"

In [ ]:
!mlflow ui \
    --host 0.0.0.0 \
    --port 5000 \
    --backend-store-uri file:///content/mlruns \
    --allowed-hosts "*" \
    > /content/mlflow.log 2>&1 &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)

print("====================================")
print("MLflow Dashboard:")
print(public_url)
print("====================================")

In [ ]:
!find /content/mlruns -maxdepth 3 -type f | head -30

In [ ]:
!mlflow --version

In [17]:
import mlflow

mlflow.set_tracking_uri("file:///content/mlruns")

print("Tracking URI:", mlflow.get_tracking_uri())

!find /content/mlruns -maxdepth 2 -type d

Tracking URI: file:///content/mlruns
/content/mlruns
/content/mlruns/765195413584141526
/content/mlruns/765195413584141526/models
/content/mlruns/765195413584141526/f8b202a6619142daa466b5f6d4b5c870
/content/mlruns/765195413584141526/e5fd21f09ee74014bf23bc3c3cf4f81f
/content/mlruns/models
/content/mlruns/.trash


In [18]:
!pkill -f "mlflow ui" || true

^C


In [19]:
import subprocess
import time

process = subprocess.Popen(
    [
        "mlflow", "ui",
        "--host", "0.0.0.0",
        "--port", "5000",
        "--backend-store-uri", "file:///content/mlruns",
        "--allowed-hosts", "*"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("MLflow server started")

MLflow server started


In [ ]:
!curl -I http://127.0.0.1:5000

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")
print(url)

In [9]:
import mlflow
import mlflow.pytorch

mlflow.set_tracking_uri("file:///content/mlruns")

mlflow.set_experiment("Food101_EfficientNetB0")

<Experiment: artifact_location='file:///content/mlruns/672852835390998665', creation_time=1788199473782, effective_trace_archival_retention=None, experiment_id='672852835390998665', last_update_time=1788199473782, lifecycle_stage='active', name='Food101_EfficientNetB0', tags={}, trace_location=None, workspace='default'>

In [8]:
import mlflow

mlflow.set_tracking_uri("file:///content/mlruns")

experiments = mlflow.search_experiments()

for exp in experiments:
    print(
        "Experiment ID:", exp.experiment_id,
        "| Name:", exp.name
    )

Experiment ID: 672852835390998665 | Name: Food101_EfficientNetB0
Experiment ID: 0 | Name: Default


In [11]:
BATCH_SIZE=32

In [13]:
# Initial dummy values in case klug6ZzQesef was not run or kernel state lost
# These will be overwritten if klug6ZzQesef successfully executed beforehand
if 'train_losses' not in globals():
    train_losses = []
if 'train_accuracies' not in globals():
    train_accuracies = []
if 'val_losses' not in globals():
    val_losses = []
if 'val_accuracies' not in globals():
    val_accuracies = []

with mlflow.start_run(run_name="EfficientNetB0_5_Epochs"):

    # Parameters
    mlflow.log_params({
        "model": "EfficientNet-B0",
        "dataset": "Food-101",
        "epochs": 5, # Note: This parameter states 5 epochs, but training might have been for fewer.
        "batch_size": BATCH_SIZE,
        "dropout": 0.4,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "num_classes": 101,
        "backbone_frozen": True
    })

    # Determine the actual number of epochs trained based on the collected metrics
    num_logged_epochs = len(train_losses)
    if num_logged_epochs == 0:
        print("[WARNING] No training metrics found (train_losses is empty). Skipping epoch-wise metric logging.")
    else:
        # Log metrics for each epoch, up to the number of epochs actually trained
        for epoch_idx in range(num_logged_epochs):
            mlflow.log_metrics({
                "train_loss": train_losses[epoch_idx],
                "train_accuracy": train_accuracies[epoch_idx],
                "val_loss": val_losses[epoch_idx],
                "val_accuracy": val_accuracies[epoch_idx]
            }, step=epoch_idx + 1)
        print(f"[INFO] Logged metrics for {num_logged_epochs} epochs.")

        # Final metrics - log only if data exists
        mlflow.log_metrics({
            "final_train_accuracy": train_accuracies[-1],
            "final_val_accuracy": val_accuracies[-1],
            "final_train_loss": train_losses[-1],
            "final_val_loss": val_losses[-1]
        })
        print("[INFO] Logged final metrics.")

    # Check if 'model' and 'device' are defined before logging model
    if 'model' in globals() and 'device' in globals():
        import torch # Ensure torch is imported for torch.randn
        mlflow.pytorch.log_model(
            model,
            name="efficientnet_b0_food101",
            input_example=torch.randn(1, 3, 224, 224).to(device),
            serialization_format='pickle' # Set serialization format to 'pickle'
        )
        print("[INFO] Logged model.")
    else:
        print("[WARNING] 'model' or 'device' not found. Skipping model logging.")

print("✅ MLflow experiment logged successfully (with checks for missing data/variables).")

[WARNING] No training metrics found (train_losses is empty). Skipping epoch-wise metric logging.
[WARNING] 'model' or 'device' not found. Skipping model logging.
✅ MLflow experiment logged successfully (with checks for missing data/variables).


In [14]:
!find /content/mlruns -maxdepth 3 -type f | head -30

/content/mlruns/0/meta.yaml
/content/mlruns/672852835390998665/c1ecbdbf5c154ac18cba6950669d9b01/meta.yaml
/content/mlruns/672852835390998665/1013d13eb08e460b89e337a2a44c72db/meta.yaml
/content/mlruns/672852835390998665/1ad2432191e24a9bbda729c395151828/meta.yaml
/content/mlruns/672852835390998665/meta.yaml


In [15]:
experiments = mlflow.search_experiments()

for exp in experiments:
    print(exp.experiment_id, exp.name)

672852835390998665 Food101_EfficientNetB0
0 Default


In [16]:
from mlflow.tracking import MlflowClient

client = MlflowClient(
    tracking_uri="file:///content/mlruns"
)

experiment = client.get_experiment_by_name(
    "Food101_EfficientNetB0"
)

print("Experiment ID:", experiment.experiment_id)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id]
)

for run in runs:
    print("Run ID:", run.info.run_id)
    print("Metrics:", run.data.metrics)
    print("Parameters:", run.data.params)
    print("--------------------------")

Experiment ID: 672852835390998665
Run ID: 1ad2432191e24a9bbda729c395151828
Metrics: {}
Parameters: {'epochs': '5', 'model': 'EfficientNet-B0', 'batch_size': '32', 'dropout': '0.4', 'learning_rate': '0.001', 'weight_decay': '0.0001', 'backbone_frozen': 'True', 'dataset': 'Food-101', 'num_classes': '101'}
--------------------------
Run ID: 1013d13eb08e460b89e337a2a44c72db
Metrics: {}
Parameters: {'epochs': '5', 'model': 'EfficientNet-B0', 'batch_size': '32', 'dropout': '0.4', 'learning_rate': '0.001', 'weight_decay': '0.0001', 'backbone_frozen': 'True', 'dataset': 'Food-101', 'num_classes': '101'}
--------------------------
Run ID: c1ecbdbf5c154ac18cba6950669d9b01
Metrics: {}
Parameters: {}
--------------------------


In [17]:
print('[INFO] Killing any running MLflow UI processes...')
!pkill -f "mlflow ui" || true
print('[INFO] MLflow UI processes killed.')

[INFO] Killing any running MLflow UI processes...
^C
[INFO] MLflow UI processes killed.


In [18]:
import subprocess
import time

print('[INFO] Starting MLflow UI...')
process = subprocess.Popen(
    [
        "mlflow", "ui",
        "--host", "0.0.0.0",
        "--port", "5000",
        "--backend-store-uri", "file:///content/mlruns",
        "--allowed-hosts", "*"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5) # Give MLflow UI a moment to start

print('[INFO] MLflow UI started.')

[INFO] Starting MLflow UI...
[INFO] MLflow UI started.


In [19]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")

print("====================================")
print("MLflow Dashboard:")
print(url)
print("====================================")
print("Please click on the link above to access the MLflow UI. If you still encounter fetching errors, please let me know.")

MLflow Dashboard:
https://5000-m-s-kkb-usw3b2-1h1s2fa100srg-b.us-west3-2.prod.colab.dev
Please click on the link above to access the MLflow UI. If you still encounter fetching errors, please let me know.


In [20]:
import subprocess
import time
from google.colab.output import eval_js

print("[INFO] Checking MLflow UI process status...")
# Check if mlflow ui is running
pid_check = subprocess.run(['pgrep', '-f', 'mlflow ui'], capture_output=True, text=True)

if pid_check.stdout:
    print(f"[INFO] MLflow UI process found with PID: {pid_check.stdout.strip()}. Attempting to get proxy URL...")
else:
    print("[WARNING] MLflow UI process not found. Restarting...")
    # Kill any lingering mlflow ui processes (in case pgrep missed something or it's stuck)
    !pkill -f "mlflow ui" || true

    # Restart mlflow ui
    process = subprocess.Popen(
        [
            "mlflow", "ui",
            "--host", "0.0.0.0",
            "--port", "5000",
            "--backend-store-uri", "file:///content/mlruns",
            "--allowed-hosts", "*"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    time.sleep(7) # Give MLflow UI a bit more time to start
    print("[INFO] MLflow UI restarted. Getting new proxy URL...")

try:
    # Generate a new Colab proxy URL
    new_url = eval_js("google.colab.kernel.proxyPort(5000)")
    print("\n====================================")
    print("MLflow Dashboard:")
    print(new_url)
    print("====================================")
    print("Please click on this new link to access the MLflow UI. If it still doesn't open, please let me know.")
except Exception as e:
    print(f"[ERROR] Could not generate Colab proxy URL: {e}")
    print("Please ensure the MLflow UI is running and try again. You might need to restart the Colab runtime if the issue persists.")

[INFO] Checking MLflow UI process status...
[INFO] MLflow UI process found with PID: 3002. Attempting to get proxy URL...

MLflow Dashboard:
https://5000-m-s-kkb-usw3b2-1h1s2fa100srg-b.us-west3-2.prod.colab.dev
Please click on this new link to access the MLflow UI. If it still doesn't open, please let me know.
